# CMFD: Coarse Mesh Finite-Difference Acceleration

![Fuel-water quarter-domain with reflecting and vacuum boundaries](images/fuel_water_quarter.png)

This tutorial compares unaccelerated power iteration with coarse-mesh finite-difference (CMFD) acceleration for the same two-group eigenvalue problem.

## Problem setup

The 14 cm by 14 cm quarter-domain contains a 10 cm by 10 cm fuel region surrounded by water on its top and right. The left and bottom boundaries are reflecting, and the outer boundaries are vacuum. The two-group cross sections come from the OpenSn regression suite.

This weakly coupled eigenvalue iteration contains a slowly converging spatial error mode. CMFD represents that mode on an aggregated mesh and applies a low-order correction after each transport update.

In [ ]:
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.solver import (
    CMFDAcceleration,
    DiscreteOrdinatesProblem,
    PowerIterationKEigenSolver,
)
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
repo_root = next(
    path for path in (tutorial_dir, *tutorial_dir.parents)
    if (path / "test/assets/xs").is_dir()
)

## Build the transport problem

The mesh uses a 2 by 2 KBA partition, so run the generated script with four MPI processes. Both solves use one transport sweep per power iteration. Rebuilding the problem ensures that the baseline and accelerated calculations start from the same state.

In [ ]:
def load_xs(filename):
    xs = MultiGroupXS()
    xs.LoadFromOpenSn(str(repo_root / "test/assets/xs" / filename))
    return xs


def make_problem():
    nodes = [0.5 * i for i in range(29)]
    partitioner = KBAGraphPartitioner(
        nx=2, ny=2, xcuts=[7.0], ycuts=[7.0]
    )
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes], partitioner=partitioner
    ).Execute()
    mesh.SetOrthogonalBoundaries()
    mesh.SetUniformBlockID(0)
    fuel_region = RPPLogicalVolume(
        xmin=-1.0, xmax=10.0, ymin=-1.0, ymax=10.0, infz=True
    )
    mesh.SetBlockIDFromLogicalVolume(fuel_region, 1, True)

    quadrature = GLCProductQuadrature2DXY(
        n_polar=4, n_azimuthal=8, scattering_order=1
    )
    return DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=2,
        groupsets=[
            {
                "groups_from_to": (0, 1),
                "angular_quadrature": quadrature,
                "inner_linear_method": "petsc_richardson",
                "l_max_its": 1,
                "l_abs_tol": 1.0e-10,
            }
        ],
        xs_map=[
            {"block_ids": [0], "xs": load_xs("xs_water_g2.xs")},
            {"block_ids": [1], "xs": load_xs("xs_fuel_g2.xs")},
        ],
        boundary_conditions=[
            {"name": "xmin", "type": "reflecting"},
            {"name": "ymin", "type": "reflecting"},
            {"name": "xmax", "type": "vacuum"},
            {"name": "ymax", "type": "vacuum"},
        ],
        options={
            "verbose_inner_iterations": False,
            "verbose_outer_iterations": False,
        },
    )

## Compare power iteration and CMFD

The baseline passes no acceleration object to the eigenvalue solver. The accelerated case adds `CMFDAcceleration`. `local_aggregation` combines nearby fine cells owned by each MPI rank, while `aggregation_size=4` targets about four fine cells per coarse cell. `relaxation=1.0` applies the full accepted correction.

In [ ]:
def solve(use_cmfd):
    problem = make_problem()
    solver_options = {
        "problem": problem,
        "k_tol": 1.0e-8,
        "max_iters": 400,
    }
    if use_cmfd:
        solver_options["acceleration"] = CMFDAcceleration(
            problem=problem,
            coarse_mesh="local_aggregation",
            aggregation_size=4,
            relaxation=1.0,
            balance_residual_tolerance=1.0e-7,
        )

    solver = PowerIterationKEigenSolver(**solver_options)
    comm.Barrier()
    start = MPI.Wtime()
    solver.Initialize()
    solver.Execute()
    comm.Barrier()
    elapsed = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)
    return (
        solver.GetEigenvalue(),
        solver.GetNumPowerIterations(),
        solver.GetNumSweeps(),
        elapsed,
    )


unaccelerated_k, unaccelerated_iterations, unaccelerated_sweeps, unaccelerated_time = solve(False)
cmfd_k, cmfd_iterations, cmfd_sweeps, cmfd_time = solve(True)

## Interpret the metrics

Agreement in $k_{\mathrm{eff}}$ checks that CMFD converges to the transport solution. Power iterations and transport sweeps measure deterministic convergence work. Wall time includes CMFD setup and solves and is shown for context; it varies by machine.

In [ ]:
k_difference = abs(cmfd_k - unaccelerated_k)
speedup = unaccelerated_time / cmfd_time

if rank == 0:
    print(f"Unaccelerated CMFD comparison k-effective={unaccelerated_k:.12e}")
    print(f"CMFD k-effective={cmfd_k:.12e}")
    print(f"CMFD k-effective difference={k_difference:.12e}")
    print(f"Unaccelerated CMFD comparison power iteration count={unaccelerated_iterations}")
    print(f"CMFD power iteration count={cmfd_iterations}")
    print(f"Unaccelerated CMFD comparison sweeps={unaccelerated_sweeps}")
    print(f"CMFD sweeps={cmfd_sweeps}")
    print(f"Unaccelerated CMFD comparison wall time (s)={unaccelerated_time:.6f}")
    print(f"CMFD wall time (s)={cmfd_time:.6f}")
    print(f"CMFD speedup={speedup:.6f}")

assert k_difference < 1.0e-6
assert cmfd_iterations < unaccelerated_iterations
assert cmfd_sweeps < unaccelerated_sweeps

A representative four-process run gives:

| Solve | $k_{\mathrm{eff}}$ | Power iterations | Transport sweeps | Wall time (s) |
|---|---:|---:|---:|---:|
| No acceleration | 0.59638182 | 264 | 264 | 0.189 |
| CMFD | 0.59638194 | 29 | 29 | 0.079 |

The eigenvalues differ by $1.2\times10^{-7}$. CMFD removes the slowly converging spatial mode and reduces both power iterations and sweeps by about 89%. The timing values are illustrative rather than regression metrics.

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()